# Notebook 1: Data Processing

This notebook handles all data cleaning, filtering, imputation, and feature engineering steps. The output is a clean CSV file (`data/processed_data.csv`) that feeds directly into the EDA and modelling notebooks.

**Dataset:** Our World in Data — World Energy Consumption  
**Target variable:** `primary_energy_consumption` (TWh)  
**Scope:** 218 countries, 1990–2022

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Raw Data

In [ ]:
df_raw = pd.read_csv('../data/World_Energy_Consumption.csv')
print(f"Raw dataset shape: {df_raw.shape}")
print(f"Countries/regions: {df_raw['country'].nunique()}")
print(f"Year range: {df_raw['year'].min()} – {df_raw['year'].max()}")
df_raw.head(3)

## 2. Filter to Usable Scope

Two filters are applied:
1. **Temporal:** Keep 1990–2022 only. Pre-1990 data is too sparse to be useful for most countries.
2. **Geographic:** Remove aggregate/regional entities (e.g. `ASEAN (Ember)`, `World`) using the ISO code column — real countries all have a valid 3-letter code.

In [ ]:
# Temporal filter
df = df_raw[df_raw['year'] >= 1990].copy()

# Remove aggregate regions — they have no ISO code or use OWID prefixes
df = df[df['iso_code'].notna() & ~df['iso_code'].str.startswith('OWID', na=False)]

# Keep only rows where the target is present
df = df[df['primary_energy_consumption'].notna()].reset_index(drop=True)

print(f"After filtering: {df.shape[0]} rows, {df['country'].nunique()} countries")

## 3. Select Features

Columns are chosen based on two criteria:
- **Missingness < 35%** in the filtered dataset
- **No leakage:** columns that are arithmetic sub-components of the target (e.g. `fossil_fuel_consumption`, r=0.998) are excluded.

The selected features represent four conceptual groups: scale (population, GDP), electricity mix (generation, share by source), fuel production, and temporal trend (year).

In [ ]:
FEATURE_COLS = [
    # Identifiers
    'country', 'year', 'iso_code',
    # Scale
    'population', 'gdp',
    # Electricity
    'electricity_generation',
    'renewables_electricity', 'solar_electricity',
    'wind_electricity', 'hydro_electricity', 'nuclear_electricity',
    # Electricity share
    'renewables_share_elec', 'fossil_share_elec',
    'solar_share_elec', 'wind_share_elec',
    'hydro_share_elec', 'nuclear_share_elec',
    # Fuel production
    'coal_production', 'oil_production', 'gas_production',
    # Target
    'primary_energy_consumption'
]

df = df[FEATURE_COLS].copy()

# Quick missingness overview
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
print("Missing values (%) per column:")
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

## 4. Feature Engineering

A few derived features are added before imputation so the imputer can use them as signals:
- `energy_per_capita`: normalises consumption for country size
- `renewables_share_gen`: computed from absolute values as a cross-check on the reported share columns
- `log_gdp`, `log_population`, `log_energy`: log transforms to reduce right skew
- `year_norm`: year scaled to [0, 1] so it doesn't dominate distance-based imputation

In [ ]:
# Per-capita energy (handle div-by-zero for very small populations)
df['energy_per_capita'] = df['primary_energy_consumption'] / df['population'].replace(0, np.nan) * 1e6

# Renewables fraction from absolute generation values
df['renewables_share_gen'] = df['renewables_electricity'] / df['electricity_generation'].replace(0, np.nan)

# Log transforms — add 1 to handle zeros
for col in ['gdp', 'population', 'primary_energy_consumption', 'electricity_generation']:
    df[f'log_{col}'] = np.log1p(df[col])

# Normalised year
df['year_norm'] = (df['year'] - 1990) / (2022 - 1990)

print("New features added:")
new_cols = ['energy_per_capita','renewables_share_gen','log_gdp','log_population',
            'log_primary_energy_consumption','log_electricity_generation','year_norm']
print(df[new_cols].describe().round(2))

## 5. Imputation

KNN imputation (k=5) is used for all numeric columns. This is preferable to simple median imputation here because the data has strong country-level structure — nearby countries in feature space tend to have similar energy profiles.

**Note:** Identifiers (`country`, `year`, `iso_code`) are held out and re-attached after imputation.

In [ ]:
id_cols = ['country', 'year', 'iso_code']
numeric_cols = [c for c in df.columns if c not in id_cols]

imputer = KNNImputer(n_neighbors=5)
df_imputed_array = imputer.fit_transform(df[numeric_cols])
df_imputed = pd.DataFrame(df_imputed_array, columns=numeric_cols)

# Re-attach identifiers
df_clean = pd.concat([df[id_cols].reset_index(drop=True), df_imputed], axis=1)

print(f"Missing values after imputation: {df_clean.isnull().sum().sum()}")
print(f"Final shape: {df_clean.shape}")

## 6. Sanity Checks

In [ ]:
# Check that imputed values are in a plausible range
print("Target variable range (should be > 0):")
print(f"  min: {df_clean['primary_energy_consumption'].min():.2f} TWh")
print(f"  max: {df_clean['primary_energy_consumption'].max():.2f} TWh")
print(f"  median: {df_clean['primary_energy_consumption'].median():.2f} TWh")

# Check share columns are bounded [0, 100]
share_cols = [c for c in df_clean.columns if 'share' in c]
print("\nShare columns — any values outside [0, 100]?")
for c in share_cols:
    n_bad = ((df_clean[c] < 0) | (df_clean[c] > 100)).sum()
    if n_bad > 0:
        print(f"  {c}: {n_bad} out-of-range values — clipping")
        df_clean[c] = df_clean[c].clip(0, 100)
    else:
        print(f"  {c}: OK")

## 7. Train / Validation / Test Split

A **time-based split** is used to avoid look-ahead bias. Models are never shown future data during training.

In [ ]:
train = df_clean[df_clean['year'] <= 2012].copy()
val   = df_clean[(df_clean['year'] >= 2013) & (df_clean['year'] <= 2017)].copy()
test  = df_clean[df_clean['year'] >= 2018].copy()

print(f"Train set (1990-2012): {len(train):,} rows")
print(f"Validation set (2013-2017): {len(val):,} rows")
print(f"Test set (2018-2022): {len(test):,} rows")

## 8. Save Processed Data

In [ ]:
df_clean.to_csv('../data/processed_data.csv', index=False)
train.to_csv('../data/train.csv', index=False)
val.to_csv('../data/val.csv', index=False)
test.to_csv('../data/test.csv', index=False)

print("Saved:")
print("  data/processed_data.csv  — full cleaned dataset")
print("  data/train.csv           — training set (1990-2012)")
print("  data/val.csv             — validation set (2013-2017)")
print("  data/test.csv            — test set (2018-2022)")